# 12 — Current WC2026 paper picks

This notebook applies the strategy from notebook 11 to the current World Cup 2026 odds ledger.

It answers the practical question:

**Which current matches should the forecasting system track more closely?**

Important notes:
- This is paper-only.
- Real-money stake is always `0`.
- It does not call external APIs if the current odds ledger already exists.
- To refresh odds, run `04_current_wc2026_odds_ledger.ipynb` first.

Input:
- `data/processed/training_features.csv`
- `data/processed/join_train_backtest/training_features_joined_market.csv`
- `data/processed/current_wc2026_odds_ledger/wc2026_current_odds_ledger.csv`
- optional: `data/processed/11_market_anchored_correction/decision_report.json`

Output:
- `data/processed/12_current_wc2026_paper_picks/current_match_predictions.csv`
- `data/processed/12_current_wc2026_paper_picks/current_paper_picks.csv`
- `data/processed/12_current_wc2026_paper_picks_report_bundle.zip`


In [ ]:
# Cell 1. Imports and helpers.

from pathlib import Path
import json
import re
import zipfile
import warnings

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("data")
PROCESSED_DIR = DATA_DIR / "processed"

OUT_DIR = PROCESSED_DIR / "12_current_wc2026_paper_picks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP_UTC = pd.Timestamp.utcnow().isoformat()
SEED = 42

TEAM_REPLACEMENTS = {
    "USA": "United States",
    "US": "United States",
    "Korea Republic": "South Korea",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic",
    "Bosnia and Herzegovina": "Bosnia & Herzegovina",
    "Ivory Coast": "Côte d'Ivoire",
    "Cote d Ivoire": "Côte d'Ivoire",
}

def norm_team(name):
    if pd.isna(name):
        return ""
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    return TEAM_REPLACEMENTS.get(name, name)

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False, default=str)
    return path

def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10 ** (-(rating_a - rating_b) / 400.0))

def normalize_rows(arr):
    arr = np.asarray(arr, dtype=float)
    arr = np.clip(arr, 1e-6, 1.0)
    return arr / arr.sum(axis=1, keepdims=True)

def make_logit(C=0.5):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=3000,
            C=C,
            random_state=SEED,
        )),
    ])

def make_poisson(alpha=1.0):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", PoissonRegressor(
            alpha=alpha,
            max_iter=1000,
        )),
    ])

def poisson_score_probs(lambda_home, lambda_away, max_goals=10):
    import math

    lambda_home = float(np.clip(lambda_home, 0.05, 8.0))
    lambda_away = float(np.clip(lambda_away, 0.05, 8.0))

    home_goals = np.arange(max_goals + 1)
    away_goals = np.arange(max_goals + 1)

    home_probs = np.exp(-lambda_home) * np.power(
        lambda_home,
        home_goals,
    ) / np.array([math.factorial(int(x)) for x in home_goals])

    away_probs = np.exp(-lambda_away) * np.power(
        lambda_away,
        away_goals,
    ) / np.array([math.factorial(int(x)) for x in away_goals])

    mat = np.outer(home_probs, away_probs)
    mat = mat / mat.sum()

    p_home = float(np.tril(mat, -1).sum())
    p_draw = float(np.trace(mat))
    p_away = float(np.triu(mat, 1).sum())

    max_idx = np.unravel_index(np.argmax(mat), mat.shape)

    return {
        "p_away": p_away,
        "p_draw": p_draw,
        "p_home": p_home,
        "most_likely_home_score": int(max_idx[0]),
        "most_likely_away_score": int(max_idx[1]),
    }

def safe_div(a, b):
    if pd.isna(a) or pd.isna(b) or b == 0:
        return np.nan
    return a / b

print("Setup OK.")


In [ ]:
# Cell 2. Load required data.

training_path = PROCESSED_DIR / "training_features.csv"
joined_market_path = (
    PROCESSED_DIR
    / "join_train_backtest"
    / "training_features_joined_market.csv"
)
current_ledger_path = (
    PROCESSED_DIR
    / "current_wc2026_odds_ledger"
    / "wc2026_current_odds_ledger.csv"
)
decision_11_path = (
    PROCESSED_DIR
    / "11_market_anchored_correction"
    / "decision_report.json"
)

if not training_path.exists():
    raise FileNotFoundError("Missing training_features.csv. Run 01 first.")

if not joined_market_path.exists():
    raise FileNotFoundError(
        "Missing training_features_joined_market.csv. Run 03 first."
    )

if not current_ledger_path.exists():
    raise FileNotFoundError(
        "Missing wc2026_current_odds_ledger.csv. Run 04 first."
    )

training = pd.read_csv(training_path)
joined_market = pd.read_csv(joined_market_path)
current_ledger = pd.read_csv(current_ledger_path)

training["date"] = pd.to_datetime(training["date"], errors="coerce")
joined_market["date"] = pd.to_datetime(joined_market["date"], errors="coerce")
current_ledger["commence_time"] = pd.to_datetime(
    current_ledger["commence_time"],
    utc=True,
    errors="coerce",
)

for df in [training, joined_market, current_ledger]:
    if "home_team" in df.columns:
        df["home_team"] = df["home_team"].map(norm_team)
    if "away_team" in df.columns:
        df["away_team"] = df["away_team"].map(norm_team)

training = training.dropna(subset=["date", "outcome"]).copy()
joined_market = joined_market.dropna(subset=["date", "outcome"]).copy()
current_ledger = current_ledger.dropna(
    subset=["commence_time", "home_team", "away_team"],
).copy()

training["outcome"] = training["outcome"].astype(int)
joined_market["outcome"] = joined_market["outcome"].astype(int)

# Keep latest snapshot per event.
if "run_timestamp_utc" in current_ledger.columns:
    current_ledger["run_timestamp_utc"] = pd.to_datetime(
        current_ledger["run_timestamp_utc"],
        utc=True,
        errors="coerce",
    )

    current = (
        current_ledger
        .sort_values(["event_id", "run_timestamp_utc"])
        .groupby("event_id", as_index=False)
        .tail(1)
        .copy()
    )
else:
    current = current_ledger.copy()

print("training:", training.shape)
print("joined_market:", joined_market.shape)
print("current fixtures:", current.shape)

display(current.head())


In [ ]:
# Cell 3. Recompute latest team state from historical training data.

# We rebuild Elo/recent/rest state so current WC2026 features are created
# in the same style as training_features.

K = 30.0
ratings = {}
recent = {}
last_date = {}

hist = training.sort_values("date").copy()

for _, r in hist.iterrows():
    date = r["date"]
    h = r["home_team"]
    a = r["away_team"]
    hg = float(r["home_score"])
    ag = float(r["away_score"])

    rh = ratings.get(h, 1500.0)
    ra = ratings.get(a, 1500.0)

    exp_h = expected_score(rh, ra)

    actual_h = 1.0 if hg > ag else 0.0 if hg < ag else 0.5

    ratings[h] = rh + K * (actual_h - exp_h)
    ratings[a] = ra + K * ((1.0 - actual_h) - (1.0 - exp_h))

    hp, ap = (3, 0) if hg > ag else (0, 3) if hg < ag else (1, 1)

    recent.setdefault(h, []).append({"gf": hg, "ga": ag, "points": hp})
    recent.setdefault(a, []).append({"gf": ag, "ga": hg, "points": ap})

    last_date[h] = date
    last_date[a] = date

def rolling_stats(team):
    hist_team = recent.get(team, [])[-5:]

    if not hist_team:
        return {
            "points": 0.0,
            "gf": 0.0,
            "ga": 0.0,
            "n": 0,
        }

    return {
        "points": float(np.mean([x["points"] for x in hist_team])),
        "gf": float(np.mean([x["gf"] for x in hist_team])),
        "ga": float(np.mean([x["ga"] for x in hist_team])),
        "n": len(hist_team),
    }

team_state = pd.DataFrame([{
    "team": team,
    "elo": rating,
    "last_date": last_date.get(team),
    "recent_matches": len(recent.get(team, [])[-5:]),
} for team, rating in ratings.items()])

team_state.to_csv(OUT_DIR / "latest_team_state.csv", index=False)

print("teams in state:", len(team_state))
display(team_state.sort_values("elo", ascending=False).head(20))


In [ ]:
# Cell 4. Build current match feature rows.

feature_rows = []

for _, r in current.iterrows():
    kickoff = r["commence_time"]
    kickoff_naive = kickoff.tz_convert(None) if getattr(kickoff, "tzinfo", None) else kickoff

    h = r["home_team"]
    a = r["away_team"]

    rh = ratings.get(h, 1500.0)
    ra = ratings.get(a, 1500.0)

    exp_h = expected_score(rh, ra)

    hs = rolling_stats(h)
    aas = rolling_stats(a)

    h_last = last_date.get(h)
    a_last = last_date.get(a)

    rest_h = (
        (kickoff_naive - h_last).days
        if h_last is not None
        and not pd.isna(h_last)
        else np.nan
    )

    rest_a = (
        (kickoff_naive - a_last).days
        if a_last is not None
        and not pd.isna(a_last)
        else np.nan
    )

    row = {
        "event_id": r.get("event_id", ""),
        "date": kickoff_naive,
        "commence_time": r["commence_time"],
        "home_team": h,
        "away_team": a,
        "is_neutral": 1,
        "is_home_not_neutral": 0,
        "neutral": True,
        "elo_home_pre": rh,
        "elo_away_pre": ra,
        "elo_diff": rh - ra,
        "elo_home_expected": exp_h,
        "home_form_points_per_match_5": hs["points"],
        "away_form_points_per_match_5": aas["points"],
        "diff_form_points_per_match_5": hs["points"] - aas["points"],
        "home_goals_for_avg_5": hs["gf"],
        "away_goals_for_avg_5": aas["gf"],
        "diff_goals_for_avg_5": hs["gf"] - aas["gf"],
        "home_goals_against_avg_5": hs["ga"],
        "away_goals_against_avg_5": aas["ga"],
        "diff_goals_against_avg_5": hs["ga"] - aas["ga"],
        "home_recent_matches": hs["n"],
        "away_recent_matches": aas["n"],
        "home_rest_days": rest_h,
        "away_rest_days": rest_a,
        "diff_rest_days": (
            rest_h - rest_a
            if not pd.isna(rest_h) and not pd.isna(rest_a)
            else np.nan
        ),
        # Current market odds/probs.
        "market_p_away": r.get("market_p_away_devig", np.nan),
        "market_p_draw": r.get("market_p_draw_devig", np.nan),
        "market_p_home": r.get("market_p_home_devig", np.nan),
        "best_away_odds_24h": r.get("best_away_odds", np.nan),
        "best_draw_odds_24h": r.get("best_draw_odds", np.nan),
        "best_home_odds_24h": r.get("best_home_odds", np.nan),
        "avg_away_odds": r.get("avg_away_odds", np.nan),
        "avg_draw_odds": r.get("avg_draw_odds", np.nan),
        "avg_home_odds": r.get("avg_home_odds", np.nan),
        "market_margin_avg": r.get("market_margin_avg", np.nan),
        "n_bookmakers": r.get("n_bookmakers", np.nan),
    }

    feature_rows.append(row)

current_features = pd.DataFrame(feature_rows)

current_features.to_csv(
    OUT_DIR / "current_match_features.csv",
    index=False,
)

print("current_features:", current_features.shape)
display(current_features.head())


In [ ]:
# Cell 5. Train football-only model and predict current matches.

exclude = {
    "date",
    "home_team",
    "away_team",
    "tournament",
    "outcome",
    "home_score",
    "away_score",
}

football_features = [
    col for col in training.columns
    if col not in exclude
    and pd.api.types.is_numeric_dtype(training[col])
    and col in current_features.columns
]

football_model = make_logit(C=0.5)

football_model.fit(
    training[football_features],
    training["outcome"],
)

raw = football_model.predict_proba(
    current_features[football_features],
)

football_p = np.zeros((len(current_features), 3))

for i, cls in enumerate(football_model.named_steps["model"].classes_):
    football_p[:, int(cls)] = raw[:, i]

current_features["football_p_away"] = football_p[:, 0]
current_features["football_p_draw"] = football_p[:, 1]
current_features["football_p_home"] = football_p[:, 2]

print("football features:", len(football_features))
display(current_features[[
    "home_team",
    "away_team",
    "football_p_away",
    "football_p_draw",
    "football_p_home",
]].head())


In [ ]:
# Cell 6. Train score models and predict current score probabilities.

score_features = football_features

home_score_model = make_poisson(alpha=1.0)
away_score_model = make_poisson(alpha=1.0)

home_score_model.fit(
    training[score_features],
    training["home_score"].clip(lower=0),
)

away_score_model.fit(
    training[score_features],
    training["away_score"].clip(lower=0),
)

lambda_home = home_score_model.predict(
    current_features[score_features],
)

lambda_away = away_score_model.predict(
    current_features[score_features],
)

lambda_home = np.clip(lambda_home, 0.05, 8.0)
lambda_away = np.clip(lambda_away, 0.05, 8.0)

score_probs = []
most_likely_scores = []

for lh, la in zip(lambda_home, lambda_away):
    out = poisson_score_probs(lh, la, max_goals=10)

    score_probs.append([
        out["p_away"],
        out["p_draw"],
        out["p_home"],
    ])

    most_likely_scores.append((
        out["most_likely_home_score"],
        out["most_likely_away_score"],
    ))

score_probs = np.asarray(score_probs)

current_features["lambda_home"] = lambda_home
current_features["lambda_away"] = lambda_away
current_features["score_p_away"] = score_probs[:, 0]
current_features["score_p_draw"] = score_probs[:, 1]
current_features["score_p_home"] = score_probs[:, 2]
current_features["most_likely_home_score"] = [x[0] for x in most_likely_scores]
current_features["most_likely_away_score"] = [x[1] for x in most_likely_scores]

display(current_features[[
    "home_team",
    "away_team",
    "lambda_home",
    "lambda_away",
    "most_likely_home_score",
    "most_likely_away_score",
    "score_p_away",
    "score_p_draw",
    "score_p_home",
]].head())


In [ ]:
# Cell 7. Train outcome model with T24 market features and predict current matches.

# This mirrors notebook 10 but uses all joined historical market data.
# It is small-data and noisy, so we only use it as a correction signal.

market_like_cols = [
    col for col in joined_market.columns
    if col.startswith("T_minus_24h_")
    and pd.api.types.is_numeric_dtype(joined_market[col])
]

# Map current columns to T_minus_24h-style columns.
current_for_outcome = current_features.copy()

rename_to_t24 = {
    "market_p_away": "T_minus_24h_market_p_away_devig",
    "market_p_draw": "T_minus_24h_market_p_draw_devig",
    "market_p_home": "T_minus_24h_market_p_home_devig",
    "best_away_odds_24h": "T_minus_24h_best_away_odds",
    "best_draw_odds_24h": "T_minus_24h_best_draw_odds",
    "best_home_odds_24h": "T_minus_24h_best_home_odds",
    "avg_away_odds": "T_minus_24h_avg_away_odds",
    "avg_draw_odds": "T_minus_24h_avg_draw_odds",
    "avg_home_odds": "T_minus_24h_avg_home_odds",
    "market_margin_avg": "T_minus_24h_market_margin_avg",
    "n_bookmakers": "T_minus_24h_n_bookmakers",
}

for src, dst in rename_to_t24.items():
    if src in current_for_outcome.columns:
        current_for_outcome[dst] = current_for_outcome[src]

joined_numeric = [
    col for col in joined_market.columns
    if pd.api.types.is_numeric_dtype(joined_market[col])
]

exclude_joined = {
    "outcome",
    "home_score",
    "away_score",
}

candidate_features = [
    col for col in joined_numeric
    if col not in exclude_joined
    and (
        col in football_features
        or col.startswith("T_minus_24h_")
    )
    and col in current_for_outcome.columns
]

outcome_model = make_logit(C=0.1)

outcome_model.fit(
    joined_market[candidate_features],
    joined_market["outcome"],
)

raw = outcome_model.predict_proba(
    current_for_outcome[candidate_features],
)

outcome_p = np.zeros((len(current_for_outcome), 3))

for i, cls in enumerate(outcome_model.named_steps["model"].classes_):
    outcome_p[:, int(cls)] = raw[:, i]

current_features["outcome_model_p_away"] = outcome_p[:, 0]
current_features["outcome_model_p_draw"] = outcome_p[:, 1]
current_features["outcome_model_p_home"] = outcome_p[:, 2]

print("outcome correction features:", len(candidate_features))
display(current_features[[
    "home_team",
    "away_team",
    "outcome_model_p_away",
    "outcome_model_p_draw",
    "outcome_model_p_home",
]].head())


In [ ]:
# Cell 8. Apply market-anchored correction from notebook 11.

# Default from the best result we saw in notebook 11.
best_blend_params = {
    "alpha_football": 0.10,
    "alpha_score": -0.25,
    "alpha_outcome_model": 0.40,
}

if decision_11_path.exists():
    try:
        with open(decision_11_path, "r", encoding="utf-8") as f:
            decision_11 = json.load(f)

        if "best_blend_params" in decision_11:
            best_blend_params = decision_11["best_blend_params"]

        print("Loaded params from 11:", best_blend_params)
    except Exception as exc:
        print("Could not read decision 11. Using defaults.", exc)
else:
    print("Decision 11 not found. Using defaults:", best_blend_params)

market_p = current_features[[
    "market_p_away",
    "market_p_draw",
    "market_p_home",
]].to_numpy()

football_p = current_features[[
    "football_p_away",
    "football_p_draw",
    "football_p_home",
]].to_numpy()

score_p = current_features[[
    "score_p_away",
    "score_p_draw",
    "score_p_home",
]].to_numpy()

outcome_p = current_features[[
    "outcome_model_p_away",
    "outcome_model_p_draw",
    "outcome_model_p_home",
]].to_numpy()

p_final = (
    market_p
    + best_blend_params["alpha_football"] * (football_p - market_p)
    + best_blend_params["alpha_score"] * (score_p - market_p)
    + best_blend_params["alpha_outcome_model"] * (outcome_p - market_p)
)

p_final = normalize_rows(p_final)

current_features["final_p_away"] = p_final[:, 0]
current_features["final_p_draw"] = p_final[:, 1]
current_features["final_p_home"] = p_final[:, 2]

current_features["predicted_outcome"] = np.select(
    [
        p_final.argmax(axis=1) == 0,
        p_final.argmax(axis=1) == 1,
        p_final.argmax(axis=1) == 2,
    ],
    [
        "away",
        "draw",
        "home",
    ],
)

current_features.to_csv(
    OUT_DIR / "current_match_predictions.csv",
    index=False,
)

display(current_features[[
    "commence_time",
    "home_team",
    "away_team",
    "market_p_home",
    "market_p_draw",
    "market_p_away",
    "final_p_home",
    "final_p_draw",
    "final_p_away",
    "predicted_outcome",
    "most_likely_home_score",
    "most_likely_away_score",
]].head(30))


In [ ]:
# Cell 9. Build paper pick candidates.

# Main strategy from 11:
# min_edge = 0.05
# min_ev = 0.00
# odds range = 1.8–6.0
#
# This is paper-only. No real stake.

STRATEGY_NAME = "market_anchored_edge_v2_paper"
MIN_EDGE = 0.05
MIN_EV = 0.00
MIN_ODDS = 1.80
MAX_ODDS = 6.00

pick_rows = []

for _, row in current_features.iterrows():
    for selection, cls in [
        ("away", 0),
        ("draw", 1),
        ("home", 2),
    ]:
        model_p = row[f"final_p_{selection}"]
        market_p_sel = row[f"market_p_{selection}"]
        odds = row[f"best_{selection}_odds_24h"]

        if pd.isna(model_p) or pd.isna(market_p_sel) or pd.isna(odds):
            continue

        edge = model_p - market_p_sel
        ev = model_p * odds - 1.0

        passes = (
            edge >= MIN_EDGE
            and ev >= MIN_EV
            and odds >= MIN_ODDS
            and odds <= MAX_ODDS
        )

        pick_rows.append({
            "strategy": STRATEGY_NAME,
            "paper_only": True,
            "real_money_stake_usd": 0.0,
            "commence_time": row["commence_time"],
            "home_team": row["home_team"],
            "away_team": row["away_team"],
            "selection": selection,
            "selection_class": cls,
            "best_decimal_odds": odds,
            "model_probability": model_p,
            "market_probability": market_p_sel,
            "edge": edge,
            "model_expected_value": ev,
            "passes_strategy": passes,
            "reason": (
                "passes_market_anchored_edge_v2"
                if passes
                else "does_not_pass_filters"
            ),
            "most_likely_score": (
                f"{int(row['most_likely_home_score'])}-"
                f"{int(row['most_likely_away_score'])}"
            ),
            "lambda_home": row["lambda_home"],
            "lambda_away": row["lambda_away"],
            "final_p_home": row["final_p_home"],
            "final_p_draw": row["final_p_draw"],
            "final_p_away": row["final_p_away"],
            "market_p_home": row["market_p_home"],
            "market_p_draw": row["market_p_draw"],
            "market_p_away": row["market_p_away"],
        })

all_candidates = pd.DataFrame(pick_rows)

current_paper_picks = all_candidates[
    all_candidates["passes_strategy"]
].copy()

current_paper_picks = current_paper_picks.sort_values(
    ["model_expected_value", "edge"],
    ascending=False,
)

all_candidates.to_csv(
    OUT_DIR / "current_all_selection_candidates.csv",
    index=False,
)

current_paper_picks.to_csv(
    OUT_DIR / "current_paper_picks.csv",
    index=False,
)

display(current_paper_picks)
print("paper picks:", len(current_paper_picks))


In [ ]:
# Cell 10. Create human-readable watchlist.

if len(current_paper_picks) == 0:
    watchlist = pd.DataFrame(columns=[
        "kickoff_utc",
        "match",
        "paper_selection",
        "odds",
        "model_probability",
        "market_probability",
        "edge",
        "expected_value",
        "most_likely_score",
        "real_money_stake_usd",
    ])
else:
    watchlist = current_paper_picks.copy()

    watchlist["kickoff_utc"] = watchlist["commence_time"]
    watchlist["match"] = (
        watchlist["home_team"]
        + " vs "
        + watchlist["away_team"]
    )
    watchlist["paper_selection"] = watchlist["selection"]
    watchlist["odds"] = watchlist["best_decimal_odds"]
    watchlist["expected_value"] = watchlist["model_expected_value"]

    watchlist = watchlist[[
        "kickoff_utc",
        "match",
        "paper_selection",
        "odds",
        "model_probability",
        "market_probability",
        "edge",
        "expected_value",
        "most_likely_score",
        "real_money_stake_usd",
    ]]

watchlist.to_csv(
    OUT_DIR / "human_readable_current_watchlist.csv",
    index=False,
)

display(watchlist)


In [ ]:
# Cell 11. Save report zip.

summary = {
    "run_timestamp_utc": RUN_TIMESTAMP_UTC,
    "current_matches": int(len(current_features)),
    "all_selection_candidates": int(len(all_candidates)),
    "paper_picks": int(len(current_paper_picks)),
    "strategy": STRATEGY_NAME,
    "min_edge": MIN_EDGE,
    "min_ev": MIN_EV,
    "min_odds": MIN_ODDS,
    "max_odds": MAX_ODDS,
    "blend_params": best_blend_params,
    "real_money_allowed": False,
    "note": (
        "These are paper-only watchlist picks. "
        "They are not forecasting instructions."
    ),
}

save_json(
    OUT_DIR / "summary.json",
    summary,
)

zip_path = PROCESSED_DIR / "12_current_wc2026_paper_picks_report_bundle.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:
    for file in OUT_DIR.rglob("*"):
        if file.is_file():
            zf.write(file, arcname=file.relative_to(OUT_DIR))

display(pd.DataFrame([summary]))

print("Created:", zip_path.resolve())
print("Report bundle created.")
